# RgDAAO–ESM-2 benchmark
Select **Runtime → Change runtime type → GPU**, then run the two cells.
This notebook runs the repository implementation directly. Both configurations are fixed before test evaluation.
The second cell downloads a results archive to your computer; retain it before the Colab runtime expires.
A fresh runtime is required for a new experiment; existing result folders are never overwritten.

In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path("/content/RgDAAO-ESM2")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/jjimenezgar/RgDAAO-ESM2.git", str(repo)], check=True)
os.chdir(repo)
print("Repository commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before the full benchmark."
print(torch.cuda.get_device_name(0))
for script in ("download_data.py", "build_dataset.py"):
    subprocess.run([sys.executable, "scripts/" + script], check=True)
subprocess.run([sys.executable, "-m", "rgdaao.prepare", "--input", "data/processed/variants.csv", "--output", "data/processed"], check=True)


In [ ]:
import shutil
from google.colab import files
subprocess.run([sys.executable, "scripts/smoke_test.py"], check=True)
subprocess.run(["bash", "scripts/run_benchmark.sh"], check=True)
shutil.copytree("data/processed", "results/data_audit", dirs_exist_ok=True)
# Retain selected models and predictions; intermediate optimizer checkpoints are unnecessary for inference.
for folder in Path("results").glob("esm2_*/checkpoints"):
    shutil.rmtree(folder)
archive = shutil.make_archive("/content/RgDAAO-ESM2-results", "zip", "results")
files.download(archive)


Each experiment saves `config.json`, `environment.json`, `pip-freeze.txt`, `run.json`, `history.json`, the selected checkpoint, `metrics.json` and `predictions.csv`.
`results/comparison.csv` and `results/figures/activity_predictions.{png,pdf}` are created only from completed test evaluations.
Share the downloaded archive to inspect and document the real comparison. The smoke report is a software check, not scientific performance.